单线程 asyncio 也会出现竞态条件。

只要多个协程操作共享状态，并在操作中间发生切换，就可能出错。

In [ ]:
import asyncio


counter = 0


async def increment():
    global counter

    current = counter
    await asyncio.sleep(0)
    counter = current + 1


async def main():
    await asyncio.gather(
        increment(),
        increment(),
    )

    print(counter)


asyncio.run(main())

上面案例就是一个状态更新丢失的案例，await 给了其他协程进入中间状态的机会。

### asyncio.Lock
锁可以保证同一时刻只有一个协程进入临界区：

In [ ]:
counter = 0
lock = asyncio.Lock()


async def increment():
    global counter

    async with lock:
        current = counter
        await asyncio.sleep(0)
        counter = current + 1


async def main():
    await asyncio.gather(
        increment(),
        increment(),
    )

    print(counter)


asyncio.run(main())

A 获得锁
→ A 完成读取和写入
→ A 释放锁
→ B 获得锁
→ B 完成读取和写入
→ B 释放锁

- 临界区应尽量小(即异步函数执行的耗时)
- 尽可能只锁住必须保护的共享状态

### Semaphore
Semaphore 可以允许固定数量的协程同时进入

In [ ]:
semaphore = asyncio.Semaphore(3)


async def call_api():
    async with semaphore:
        return await remote_api()


- 控制外部 API 并发；
- 保护数据库连接池；
- 避免瞬间创建过多任务；
- 实现简单的并发限流。